In [1]:
#check on December-8, 2025 by Hengcong Liu

import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split, GridSearchCV
from pathlib import Path
import os

# ============================================================
#                       单次 Grid Search
# ============================================================
def run_grid_search(target_species, risk_factor_df, weight_df, case_control_df):

    # 风险因子列
    all_risk_cols = risk_factor_df.columns.drop("gb")

    # 合并数据
    all_data = (
        risk_factor_df
        .merge(weight_df, on="gb", how="left")
        .merge(
            case_control_df[["gb", target_species]].rename(columns={target_species: "status"}),
            on="gb",
            how="left"
        )
    )

    survey_data = all_data.dropna(subset=["status"]).reset_index(drop=True)

    # train-test split（仅训练集，保持固定随机种子）
    train, _ = train_test_split(
        survey_data,
        test_size=0.25,
        random_state=42,
        stratify=survey_data["status"],  #Stratified sampling, 分层抽样，让训练集和测试集按照某一列保持相同的类别比例，可以避免正负样本比例失衡，出现训练集全是0或者没有0的情况
        shuffle=True #在划分前打乱数据的顺序，避免数据被分成两个不同使其，避免顺序性带来的偏差，提高泛化性
    )

    X_train = train[all_risk_cols].values
    y_train = train["status"].values
    w_train = train["rescaled"].values

    scale_pos_weight = sum(y_train == 0) / sum(y_train == 1)
    # XGBoost 模型
    model = xgb.XGBClassifier(
        objective="binary:logistic",  #二分类逻辑回归，输出0-1的概率，使用logistic loss（对数损失）
        eval_metric="logloss", #评估指标，对数损失
        random_state=42, #固定随机种子，保证结果的可重复性
        tree_method="hist", #推荐的tree_method，提升运行速度
        n_jobs=max(1, int(os.cpu_count() * 0.7)), #cpu的线程数，1=单线程，-1=自动使用全部cpu
        scale_pos_weight=scale_pos_weight #用于处理类型不平衡，提升少数类的权重
    )

    # 参数网格
    param_grid = {
        "n_estimators": [3000],
        "max_depth": [3, 5, 7],
        "learning_rate": [0.001, 0.005, 0.01],
        "subsample": [0.70, 0.75, 0.8],
    }

    grid = GridSearchCV(
        model,
        param_grid,
        scoring="neg_log_loss", #用对数损失评估模型，log_loss越小，模型越好，但是gridsearchCV智能最大化得分，即越大越好，所以需要给log_loss取反，及neg_log_loss
        cv=5, #5折交叉验证，提升稳定性，避免过拟合
        n_jobs=max(1, int(os.cpu_count() * 0.7)),
    )

    grid.fit(X_train, y_train, sample_weight=w_train)

    return {"species": target_species, **grid.best_params_}


# ============================================================
#                       主程序
# ============================================================
output_dir = Path('output/3_grid_search')
output_dir.mkdir(parents=True, exist_ok=True)  # 自动创建文件夹
results_list = []

# 读取数据
data = pd.read_excel('../1_clean/data_merge_all_20260110.xlsx')
data = data[~(data['host_species'].isna() & data['standard virus name'].isna())].reset_index(drop=True) #key
print(data.shape[0])

# baseline 的 case_control_df
# 要生成的列名及对应的分类字段
host_levels = {
    "species": "host_species",
    "genus": "host_genus"
}
case_control_dfs = {}  # 用字典存储结果
for level, col in host_levels.items():
    case_control_dfs[level] = (
        data.assign(presence=1)
            .pivot_table(
                index="gb",
                columns=col,
                values="presence",
                aggfunc="max",
                fill_value=0
            )
            .reset_index()
    )

# 权重
weight_df = pd.read_excel("output/2_weight/weight.xlsx")[["gb", "rescaled"]]

# 风险因子 baseline
risk_factor_df = pd.read_excel("output/1_risk_factor/risk_factor.xlsx")

# 读取目标 host
targets = ['Apodemus agrarius', 'Rattus norvegicus',  'Suncus murinus', 'Crocidura', 'Rattus']

# 批量运行
for idx, target_species in enumerate(targets, 1):
    if target_species!='Rattus' and target_species!='Crocidura':
        case_control_df=case_control_dfs['species']
    else:
        case_control_df=case_control_dfs['genus']
    record = run_grid_search(
            target_species,
            risk_factor_df,
            weight_df,
            case_control_df
        )

    results_list.append(record)
    print(f"✓ [{idx}/{len(targets)}] {target_species:30s} done.")

# 保存结果
results_df = pd.DataFrame(results_list)
results_file = output_dir / f'host_best_params.xlsx'
results_df.to_excel(results_file, index=False)

47196
✓ [1/5] Apodemus agrarius              done.
✓ [2/5] Rattus norvegicus              done.
✓ [3/5] Suncus murinus                 done.
✓ [4/5] Crocidura                      done.
✓ [5/5] Rattus                         done.
